# 04 · How the AIM policy evolves during training**Does pretraining change how conflict develops during optimization?**⚠️ `train.py` never logged `cos(g_i, g_j)`, so there is **no measured conflicttrace** on disk. What every epoch *does* record is τ, `L_magnitude` and`L_progress` — the policy's *response* to conflict, not conflict itself. Titledaccordingly.`L_magnitude` and `L_progress` differ by ~100× between backbones in absoluteterms, so they are indexed to epoch 1: the shape is what is comparable.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, ".")
import numpy as np, matplotlib.pyplot as plt
import common as C
C.apply_style()
print("results trees:")
for b, d in C.RESULTS.items():
    for tag, p in d.items():
        print(f"  {b:9s} {tag:7s} {'OK ' if p.is_dir() else 'MISSING'} {p}")

In [ ]:
TAG = "11task"
METHOD = "aim_matrix"
SMOOTH = 5               # rolling-mean window in epochs; 1 = raw
BAND = "iqr"             # "iqr" | "minmax" | "none"

tr = {}
for b in ("Uni-Mol", "GNN"):
    t = C.load_tau_trace(b, TAG, METHOD)
    if t is None:
        print(f"{b}: no tau logged"); continue
    tau = t["tau"]
    if tau.ndim == 3:                      # [epochs, n, n] -> off-diagonal stats
        off = ~np.eye(tau.shape[1], dtype=bool)
        vals = tau[:, off]
        t["mean"] = vals.mean(1)
        lo, hi = ((vals.min(1), vals.max(1)) if BAND == "minmax"
                  else (np.percentile(vals, 25, axis=1), np.percentile(vals, 75, axis=1)))
    else:                                  # scalar policy
        t["mean"] = tau; lo = hi = tau
    t["lo"], t["hi"] = lo, hi
    tr[b] = t
    print(f"{b:9s} {len(t['epoch'])} epochs   tau {t['mean'][0]:+.3f} -> {t['mean'][-1]:+.3f}")

In [ ]:
def smooth(y, w):
    if w <= 1: return y
    return np.convolve(np.concatenate([np.full(w - 1, y[0]), y]), np.ones(w) / w, "valid")

def index(y):
    f = y[np.isfinite(y)]
    return y / abs(f[0]) if f.size and f[0] != 0 else y

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for b, t in tr.items():
    c = C.BACKBONE_COLOUR[b]; ep = t["epoch"]
    axes[0].plot(ep, smooth(t["mean"], SMOOTH), color=c, lw=2, zorder=3, label=b)
    if BAND != "none":
        axes[0].fill_between(ep, t["lo"], t["hi"], color=c, alpha=.12, lw=0, zorder=2)
    axes[1].plot(ep, smooth(index(t["l_mag"]), SMOOTH), color=c, lw=2, zorder=3, label=b)
    axes[2].plot(ep, smooth(index(t["l_prog"]), SMOOTH), color=c, lw=2, zorder=3, label=b)

axes[0].axhline(0, color=C.AXIS, lw=1)
axes[0].set_ylabel(r"mean off-diagonal $\tau$")
axes[0].set_title(f"Learned threshold $\\tau$  (band: {BAND})", color=C.INK, loc="left", pad=8)
axes[1].set_yscale("log"); axes[1].set_ylabel("L_magnitude (indexed)")
axes[1].set_title(r"Magnitude penalty $(\|g^*\|-\Sigma_i\|g_i\|)^2$", color=C.INK, loc="left", pad=8)
axes[2].set_ylabel("L_progress (indexed)")
axes[2].set_title(r"Progress term $-\Sigma_i \alpha_i \langle g^*, g_i\rangle$", color=C.INK, loc="left", pad=8)
for ax in axes:
    ax.set_xlabel("epoch")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.grid(True, zorder=0); ax.set_axisbelow(True)
fig.suptitle("How the AIM policy evolves during training", x=.01, y=.98, ha="left",
             fontsize=15, fontweight="bold", color=C.INK)
fig.text(.01, .915, f"{TAG} · {METHOD} · Uni-Mol (pretrained) vs GNN (from scratch)"
         + (f" · {SMOOTH}-epoch rolling mean" if SMOOTH > 1 else ""),
         fontsize=9.5, color=C.MUTED, ha="left")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper right", bbox_to_anchor=(.99, .99), ncol=len(l))
fig.text(.01, .02, "The policy's response to conflict, not conflict itself: "
         "cos(g_i, g_j) is never logged by train.py.", fontsize=7.5, color=C.MUTED)
fig.subplots_adjust(left=.055, right=.985, top=.80, bottom=.16, wspace=.26)
C.save(fig, f"policy_evolution_{TAG}", TAG); plt.show()

## Learned τ as a heatmap (at the selected epoch)Free — τ is in every AIM `history.json`, no measurement needed.

In [ ]:
if TAG != "2task":
    fig, axes = plt.subplots(1, len(tr), figsize=(5.4 * len(tr), 5.0))
    axes = np.atleast_1d(axes)
    tasks = C.load_runs(list(tr)[0], TAG)["tasks"]
    mats = {b: np.asarray(t["best"]["tau"], dtype=float) for b, t in tr.items()}
    off = ~np.eye(len(tasks), dtype=bool)
    vmax = max(abs(m[off]).max() for m in mats.values())
    for ax, (b, M) in zip(axes, mats.items()):
        im = C.heatmap(ax, M, tasks, C.cmap("diverging"), -vmax, vmax,
                       f"{b}   (epoch {tr[b]['best']['epoch']})", "{:+.3f}", annotate=True)
    cb = fig.colorbar(im, ax=list(axes), fraction=.035, pad=.02)
    cb.outline.set_visible(False); cb.ax.tick_params(length=0, colors=C.MUTED, labelsize=8)
    cb.set_label(r"learned $\tau_{ij}$", color=C.SECONDARY, fontsize=8.5)
    fig.suptitle(r"Learned threshold matrix $\tau_{ij}$ at the selected epoch",
                 x=.01, y=.98, ha="left", fontsize=15, fontweight="bold", color=C.INK)
    C.save(fig, f"tau_heatmap_{TAG}", TAG); plt.show()
else:
    print("2-task tau is 2x2 — read it off the w_ij figure in notebook 05 instead.")